# GA4 E-Commerce Funnel Analysis
**Dataset:** `bigquery-public-data.ga4_obfuscated_sample_ecommerce`  
**Stack:** BigQuery (SQL) → Pandas → Plotly  
**Date range:** Nov 2020 – Jan 2021 (full public dataset)

## What this notebook covers

| Section | Analysis |
|---|---|
| 1 | Setup & config |
| 2 | Schema exploration — event inventory, date range |
| 3 | **Session-scoped funnel** — conversion within a single visit |
| 4 | Revenue layer — $ impact at each funnel step |
| 5 | Device category breakdown |
| 5c | **Acquisition channel breakdown** — is the discovery loss a traffic problem? |
| 6 | Time-to-convert — median hours between steps |
| 7 | Alternative entry: shoppers funnel starting at `view_item` |
| 7c | **Product category view → cart** — where to start the product-page audit |
| 8 | Weekly cohort retention |
| 9 | Summary & hypotheses |
| 10 | **AI-powered insight generation** — LLM product brief |

> **Session vs user grain:** sections 3–7 use session-scoped funnels (each row = one visit).
> This matches how the GA4 UI reports conversion and gives sharper, more actionable numbers
> than user-scoped funnels where a past purchaser "carries over" into every future session.

## 1. Setup

In [1]:
# ── Section 1: Setup ──────────────────────────────────────────────────────────
# Suppress non-critical library warnings so notebook output stays readable.
import warnings
warnings.filterwarnings('ignore')

# BigQuery: run SQL against GA4 export tables in the cloud.
from google.cloud import bigquery
# Pandas: reshape query results into DataFrames for metrics and charts.
import pandas as pd
# Plotly: interactive funnel, bar, and heatmap visualizations.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# GCP project that owns the BigQuery job (billing + credentials).
PROJECT_ID = "project-92e3923f-0e27-4c05-b11"
# Fully qualified BigQuery dataset: GA4 obfuscated e-commerce public sample.
DATASET    = "bigquery-public-data.ga4_obfuscated_sample_ecommerce"

# Analysis window — matches the full public sample (Nov 2020 – Jan 2021).
# Widen carefully: each extra month adds ~$0.005 per query at this dataset size.
DATE_START = "20201101"
DATE_END   = "20210131"

# Client uses Application Default Credentials (gcloud auth application-default login).
client = bigquery.Client(project=PROJECT_ID)
print(f"BigQuery client ready  |  project: {PROJECT_ID}")
print(f"Date window: {DATE_START} → {DATE_END}")


BigQuery client ready  |  project: project-92e3923f-0e27-4c05-b11
Date window: 20201101 → 20210131


## 2. Schema exploration
Confirm date range and event inventory before writing analysis queries.

In [2]:
# ── Section 2a: Dataset date range ───────────────────────────────────────────
# GA4 exports one table per day: events_YYYYMMDD. _TABLE_SUFFIX is the date suffix.
# Scan all daily tables to confirm earliest/latest dates and table count.
date_range_df = client.query(f"""
SELECT
  MIN(_TABLE_SUFFIX) AS earliest_date,
  MAX(_TABLE_SUFFIX) AS latest_date,
  COUNT(DISTINCT _TABLE_SUFFIX) AS num_daily_tables
FROM `{DATASET}.events_*`
""").to_dataframe()

print("Full dataset date range:")
display(date_range_df)


Full dataset date range:


,earliest_date,latest_date,num_daily_tables
0,20201101,20210131,92


In [3]:
# ── Section 2b: Event inventory ────────────────────────────────────────────
# List every event_name in the analysis window with volume and reach.
# Use this to validate funnel step events exist before building session logic.
events_df = client.query(f"""
SELECT
  event_name,
  COUNT(*)                       AS event_count,
  COUNT(DISTINCT user_pseudo_id) AS unique_users
FROM `{DATASET}.events_*`
WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
GROUP BY event_name
ORDER BY event_count DESC
""").to_dataframe()

print(f"Events in window {DATE_START}–{DATE_END}:")
display(events_df)


Events in window 20201101–20210131:


,event_name,event_count,unique_users
0,page_view,1350428,269792
1,user_engagement,1058721,213004
2,scroll,493072,138098
3,view_item,386068,61252
4,session_start,354970,267116
5,first_visit,257462,257314
6,view_promotion,190104,102443
7,add_to_cart,58543,12545
8,begin_checkout,38757,9715
9,select_item,31007,13180


## 3. Session-scoped funnel

**Grain:** one row = one session (`user_pseudo_id` + `ga_session_id`).
A session converts at step N only if it reached step N−1 first *within the same session*.

**Why session-scoped?**
- Matches GA4 UI funnel reports
- Avoids inflating conversion by counting past purchasers who return for unrelated visits
- Step-to-step rates reflect a single coherent user intent

**Funnel:** `session_start` → `view_item` → `add_to_cart` → `begin_checkout` → `purchase`

In [4]:
# ── Section 3: Session-scoped funnel (BigQuery) ─────────────────────────────
# Grain: one row per (user_pseudo_id, ga_session_id). Steps must occur in order
# within the same visit — mirrors GA4 UI funnel reporting.
session_funnel_sql = f"""
-- base: pull funnel events and extract ga_session_id from the event_params array.
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value
     FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id'
     LIMIT 1)           AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),

-- session_events: first occurrence of each event per session (dedupe repeats).
session_events AS (
  SELECT
    user_pseudo_id,
    session_id,
    event_name,
    MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),

-- pivoted: one row per session; columns hold earliest timestamp per step.
pivoted AS (
  SELECT
    user_pseudo_id,
    session_id,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END) AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END) AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END) AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END) AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END) AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),

-- sequential: count sessions reaching each step only if prior steps happened first.
sequential AS (
  SELECT
    COUNT(*)                                                                         AS step1_sessions,
    COUNTIF(ts_view     IS NOT NULL AND ts_view     >= ts_session)                  AS step2_view_item,
    COUNTIF(ts_cart     IS NOT NULL AND ts_cart     >= ts_view
                                   AND ts_view      >= ts_session)                  AS step3_add_to_cart,
    COUNTIF(ts_checkout IS NOT NULL AND ts_checkout >= ts_cart
                                   AND ts_cart      >= ts_view
                                   AND ts_view      >= ts_session)                  AS step4_begin_checkout,
    COUNTIF(ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout
                                   AND ts_checkout  >= ts_cart
                                   AND ts_cart      >= ts_view
                                   AND ts_view      >= ts_session)                  AS step5_purchase
  FROM pivoted
  WHERE ts_session IS NOT NULL
)

SELECT * FROM sequential
"""

funnel_raw = client.query(session_funnel_sql).to_dataframe()
print("Session-scoped funnel counts:")
display(funnel_raw)


Session-scoped funnel counts:


,step1_sessions,step2_view_item,step3_add_to_cart,step4_begin_checkout,step5_purchase
0,354857,75261,14870,5205,2721


In [5]:
# ── Section 3b: Funnel metrics + Plotly visualization ─────────────────────
# Human-readable step labels aligned with GA4 funnel stage names.
step_labels = [
    "1. Session Start",
    "2. View Item",
    "3. Add to Cart",
    "4. Begin Checkout",
    "5. Purchase",
]
# Column names returned by the sequential CTE in session_funnel_sql.
step_cols = [
    "step1_sessions", "step2_view_item", "step3_add_to_cart",
    "step4_begin_checkout", "step5_purchase",
]

# Build a tidy funnel table for display and downstream LLM context (Section 10).
funnel_df = pd.DataFrame({
    "step":  step_labels,
    "sessions": [int(funnel_raw[c].iloc[0]) for c in step_cols],
})
# step_cvr: conversion from the previous step (% who advanced one stage).
funnel_df["step_cvr"]    = (funnel_df["sessions"] / funnel_df["sessions"].shift(1) * 100).round(1)
# overall_cvr: conversion from session start (% still in funnel at this step).
funnel_df["overall_cvr"] = (funnel_df["sessions"] / funnel_df["sessions"].iloc[0] * 100).round(1)
# drop_off: 100 - step_cvr — share lost between adjacent steps.
funnel_df["drop_off"]    = (100 - funnel_df["step_cvr"]).round(1)

print("Session funnel with conversion rates:")
display(funnel_df)

# Funnel chart: width of each stage = remaining sessions.
fig = px.funnel(
    funnel_df, x="sessions", y="step",
    title=f"GA4 Session-Scoped Funnel  ({DATE_START[:6]} – {DATE_END[:6]})",
    color_discrete_sequence=["#4C78A8"],
)
# Annotate each transition with drop-off % and step conversion (skip top of funnel).
for i, row in funnel_df.iterrows():
    if i == 0:
        continue
    fig.add_annotation(
        x=row["sessions"], y=row["step"],
        text=f"  ↓ {row['drop_off']}% drop-off  ({row['step_cvr']}% pass)",
        showarrow=False, xanchor="left",
        font=dict(size=11, color="#d62728"),
    )
fig.update_layout(height=450, margin=dict(l=160, r=260, t=60, b=40))
fig.show()


Session funnel with conversion rates:


,step,sessions,step_cvr,overall_cvr,drop_off
0,1. Session Start,354857,NaN,100.0,NaN
1,2. View Item,75261,21.2,21.2,78.8
2,3. Add to Cart,14870,19.8,4.2,80.2
3,4. Begin Checkout,5205,35.0,1.5,65.0
4,5. Purchase,2721,52.3,0.8,47.7


## 4. Revenue layer
Add `ecommerce.purchase_revenue` to see the **dollar impact** at the purchase step.
Also computes revenue per session entering the funnel and revenue per converting session.

In [6]:
# ── Section 4: Revenue layer ────────────────────────────────────────────────
# Restrict to sessions that completed the full sequential funnel, then sum
# ecommerce.purchase_revenue (only populated on purchase events).
revenue_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp,
    -- Revenue only present on purchase events
    ecommerce.purchase_revenue AS purchase_revenue
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT
    user_pseudo_id, session_id, event_name,
    MIN(event_timestamp)   AS first_ts,
    SUM(purchase_revenue)  AS session_revenue
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END)      AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END)      AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END)      AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END)      AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END)      AS ts_purchase,
    MAX(CASE WHEN event_name = 'purchase'       THEN session_revenue END) AS revenue
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),
-- converting_sessions: same sequential rules as Section 3 funnel.
converting_sessions AS (
  SELECT *
  FROM pivoted
  WHERE
    ts_session  IS NOT NULL
    AND ts_view     >= ts_session
    AND ts_cart     >= ts_view
    AND ts_checkout >= ts_cart
    AND ts_purchase >= ts_checkout
)
SELECT
  COUNT(*)                                AS purchasing_sessions,
  ROUND(SUM(revenue), 2)                  AS total_revenue,
  ROUND(AVG(revenue), 2)                  AS avg_order_value,
  ROUND(APPROX_QUANTILES(revenue, 100)[OFFSET(50)], 2) AS median_order_value,
  ROUND(MIN(revenue), 2)                  AS min_order_value,
  ROUND(MAX(revenue), 2)                  AS max_order_value
FROM converting_sessions
WHERE revenue IS NOT NULL AND revenue > 0
"""

revenue_df = client.query(revenue_sql).to_dataframe()
print("Revenue summary for purchasing sessions:")
display(revenue_df)

# Normalize revenue against all sessions that entered the funnel (step 1).
total_sessions = int(funnel_raw["step1_sessions"].iloc[0])
total_revenue  = float(revenue_df["total_revenue"].iloc[0])
print(f"\nRevenue per session entering funnel: ${total_revenue / total_sessions:.4f}")
print(f"Overall session-to-purchase rate:    {int(revenue_df['purchasing_sessions'].iloc[0]) / total_sessions * 100:.2f}%")


Revenue summary for purchasing sessions:


,purchasing_sessions,total_revenue,avg_order_value,median_order_value,min_order_value,max_order_value
0,2461,193678.0,78.7,55.0,3.0,1530.0



Revenue per session entering funnel: $0.5458
Overall session-to-purchase rate:    0.69%


## 5. Device category breakdown
Same session-scoped funnel, split by `device.category`.
Highlights whether mobile vs desktop differences are meaningful or within noise.

In [7]:
# ── Section 5: Device category breakdown (BigQuery) ─────────────────────────
# Same session-scoped sequential funnel as Section 3, grouped by device.category.
device_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    device.category AS device_category,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(device_category) AS device_category,
    event_name,
    MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(device_category)                                                   AS device_category,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END)               AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END)               AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END)               AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END)               AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END)               AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
)
SELECT
  device_category,
  COUNT(*)                                                                        AS step1_sessions,
  COUNTIF(ts_view     IS NOT NULL AND ts_view     >= ts_session)                 AS step2_view_item,
  COUNTIF(ts_cart     IS NOT NULL AND ts_cart     >= ts_view
                                 AND ts_view      >= ts_session)                 AS step3_add_to_cart,
  COUNTIF(ts_checkout IS NOT NULL AND ts_checkout >= ts_cart
                                 AND ts_cart      >= ts_view
                                 AND ts_view      >= ts_session)                 AS step4_begin_checkout,
  COUNTIF(ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout
                                 AND ts_checkout  >= ts_cart
                                 AND ts_cart      >= ts_view
                                 AND ts_view      >= ts_session)                 AS step5_purchase
FROM pivoted
WHERE ts_session IS NOT NULL
GROUP BY device_category
ORDER BY step1_sessions DESC
"""

device_raw = client.query(device_sql).to_dataframe()
display(device_raw)


,device_category,step1_sessions,step2_view_item,step3_add_to_cart,step4_begin_checkout,step5_purchase
0,desktop,205899,43810,8606,2985,1537
1,mobile,141079,29793,5948,2108,1123
2,tablet,7879,1658,316,112,61


In [8]:
# ── Section 5b: Device funnel chart ───────────────────────────────────────
# Map wide SQL column names to display labels for melt/pivot operations.
step_map = {
    "step1_sessions":       "1. Session Start",
    "step2_view_item":      "2. View Item",
    "step3_add_to_cart":    "3. Add to Cart",
    "step4_begin_checkout": "4. Begin Checkout",
    "step5_purchase":       "5. Purchase",
}
# Long format: one row per (device, funnel step) for grouped bar chart.
device_long = device_raw.melt(
    id_vars="device_category",
    value_vars=list(step_map.keys()),
    var_name="step_col", value_name="sessions",
)
device_long["step"] = device_long["step_col"].map(step_map)
# Denominator for % conversion: sessions at step 1 per device.
base = (
    device_long[device_long["step_col"] == "step1_sessions"]
    [["device_category","sessions"]].rename(columns={"sessions":"base"})
)
device_long = device_long.merge(base, on="device_category")
device_long["pct"] = (device_long["sessions"] / device_long["base"] * 100).round(2)

# Grouped bar: % of each device's sessions still in funnel at each step.
fig_dev = px.bar(
    device_long, x="step", y="pct", color="device_category",
    barmode="group",
    title="Session Funnel Conversion Rate by Device",
    labels={"pct": "% of Sessions", "step": "", "device_category": "Device"},
    color_discrete_map={"desktop": "#4C78A8", "mobile": "#F58518", "tablet": "#54A24B"},
    text="pct",
)
fig_dev.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_dev.update_layout(height=450, xaxis_tickangle=-20, uniformtext_minsize=8)
fig_dev.show()

# Absolute purchase counts and session-to-purchase % by device (LLM context uses this).
purchase_rates = device_long[device_long["step_col"] == "step5_purchase"][["device_category","sessions","base","pct"]]
print("\nSession-to-purchase rate by device:")
display(purchase_rates.reset_index(drop=True))



Session-to-purchase rate by device:


,device_category,sessions,base,pct
0,desktop,1537,205899,0.75
1,mobile,1123,141079,0.8
2,tablet,61,7879,0.77


## 5c. Acquisition channel breakdown — is discovery a traffic problem?

~79% of sessions never reach a product page. Two very different explanations fit
that number, and they lead to different work:

| If the miss rate is… | Then the problem is… | And the fix is… |
|---|---|---|
| **Uniform across channels** | On the site — navigation / discovery | Homepage and category navigation |
| **Concentrated in some channels** | Who is arriving — traffic quality | Ad targeting, landing pages, budget mix |

This section splits the same session-scoped funnel by channel. A **10pp spread** in
session → view-item rate between channels is used as the decision threshold.

**Grain trap — read before reusing this pattern.** `traffic_source.medium` in the
GA4 export is the **user's first-touch** medium: every session a user ever has
inherits it. Grouping a *session* funnel by it attributes a returning direct visit
to the paid campaign that first acquired the user. The session's own medium lives
in `event_params` (`key = 'medium'`), so it is taken from the earliest event in the
session that carries one. Both are returned so they can be compared.

In [9]:
# ── Section 5c: Acquisition channel breakdown (BigQuery) ─────────────────────
# Same sequential funnel as Section 3, plus revenue, grouped by BOTH the
# session-level medium and the user's first-touch medium.
channel_sql = f"""
-- base: funnel events, session id, first-touch medium (user-level), revenue.
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1)          AS session_id,
    traffic_source.medium                             AS first_touch_medium,
    event_name,
    event_timestamp,
    ecommerce.purchase_revenue                        AS purchase_revenue
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),

-- session_medium: medium on the earliest event in the session that carries one.
-- Scans every event type: medium is not guaranteed on the funnel events.
session_medium AS (
  SELECT
    user_pseudo_id, session_id,
    ARRAY_AGG(medium ORDER BY event_timestamp LIMIT 1)[OFFSET(0)] AS session_medium
  FROM (
    SELECT
      user_pseudo_id, event_timestamp,
      (SELECT ep.value.int_value    FROM UNNEST(event_params) AS ep
       WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
      (SELECT ep.value.string_value FROM UNNEST(event_params) AS ep
       WHERE ep.key = 'medium' LIMIT 1)        AS medium
    FROM `{DATASET}.events_*`
    WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
  )
  WHERE session_id IS NOT NULL AND medium IS NOT NULL
  GROUP BY user_pseudo_id, session_id
),

session_events AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(first_touch_medium) AS first_touch_medium,
    event_name,
    MIN(event_timestamp)          AS first_ts,
    SUM(purchase_revenue)         AS rev
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),

pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(first_touch_medium)                                    AS first_touch_medium,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END)   AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END)   AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END)   AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END)   AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END)   AS ts_purchase,
    MAX(CASE WHEN event_name = 'purchase'       THEN rev END)        AS revenue
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),

-- converted: flag the full sequential path once, reuse for count and revenue.
flagged AS (
  SELECT
    p.*,
    COALESCE(sm.session_medium, '(not set)') AS session_medium_label,
    (ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout AND ts_checkout >= ts_cart
       AND ts_cart >= ts_view AND ts_view >= ts_session)    AS converted
  FROM pivoted AS p
  LEFT JOIN session_medium AS sm USING (user_pseudo_id, session_id)
  WHERE ts_session IS NOT NULL
)

SELECT
  session_medium_label                                     AS session_medium,
  COALESCE(first_touch_medium, '(not set)')                AS first_touch_medium,
  COUNT(*)                                                 AS sessions,
  COUNTIF(ts_view IS NOT NULL AND ts_view >= ts_session)   AS view_item,
  COUNTIF(ts_cart IS NOT NULL AND ts_cart >= ts_view
                              AND ts_view >= ts_session)   AS add_to_cart,
  COUNTIF(converted)                                       AS purchase,
  ROUND(SUM(IF(converted, IFNULL(revenue, 0), 0)), 2)      AS revenue
FROM flagged
GROUP BY session_medium, first_touch_medium
"""

channel_raw = client.query(channel_sql).to_dataframe()

# Coverage: if most sessions have no session-level medium, fall back to first touch.
coverage = 1 - channel_raw.loc[channel_raw["session_medium"] == "(not set)", "sessions"].sum() / channel_raw["sessions"].sum()
channel_dim = "session_medium" if coverage >= 0.5 else "first_touch_medium"
print(f"Session-level medium coverage: {coverage*100:.1f}%  →  using {channel_dim}")

# Reconciliation: every session lands in exactly one channel.
assert channel_raw["sessions"].sum() == funnel_raw["step1_sessions"].iloc[0], "channel join dropped/duplicated sessions"

# How often do the two definitions disagree? This is the grain trap, quantified.
agree = channel_raw.loc[channel_raw["session_medium"] == channel_raw["first_touch_medium"], "sessions"].sum()
print(f"Sessions where session medium == first-touch medium: {agree / channel_raw['sessions'].sum() * 100:.1f}%")
display(channel_raw.pivot_table(index="first_touch_medium", columns="session_medium",
                                values="sessions", aggfunc="sum", fill_value=0))

Session-level medium coverage: 73.8%  →  using session_medium
Sessions where session medium == first-touch medium: 52.9%


session_medium,(data deleted),(none),(not set),<Other>,affiliate,cpc,email,organic,referral
first_touch_medium,,,,,,,,,
(data deleted),6315,0,4269,0,150,0,24,3177,7527
(none),0,37670,21863,0,309,1,49,10338,12132
<Other>,0,0,14261,25302,150,1,23,6479,5560
cpc,0,0,4246,0,43,7659,7,1909,1671
organic,0,0,32735,0,406,2,46,72732,15645
referral,0,0,15518,0,251,1,43,8153,38190


In [10]:
# ── Section 5d: Channel funnel metrics + chart ──────────────────────────────
metrics = ["sessions", "view_item", "add_to_cart", "purchase", "revenue"]
channel_df = (channel_raw.groupby(channel_dim, as_index=False)[metrics].sum()
              .rename(columns={channel_dim: "channel"}))

# Group channels under 1% of sessions — tiny denominators produce noisy rates.
total = channel_df["sessions"].sum()
small = channel_df["sessions"] / total < 0.01
if small.sum() > 1:
    other = channel_df[small][metrics].sum().to_frame().T
    other.insert(0, "channel", f"other ({int(small.sum())} small channels)")
    channel_df = pd.concat([channel_df[~small], other], ignore_index=True)

channel_df[metrics]            = channel_df[metrics].astype(float)
channel_df["share_pct"]        = channel_df["sessions"] / total * 100
channel_df["view_pct"]         = channel_df["view_item"] / channel_df["sessions"] * 100
channel_df["cart_from_view"]   = channel_df["add_to_cart"] / channel_df["view_item"].where(channel_df["view_item"] > 0) * 100
channel_df["purchase_pct"]     = channel_df["purchase"] / channel_df["sessions"] * 100
channel_df["rev_per_session"]  = channel_df["revenue"] / channel_df["sessions"]
channel_df["low_n"]            = channel_df["purchase"] < 30   # purchase rate too noisy to rank
channel_df = channel_df.sort_values("sessions", ascending=False).reset_index(drop=True)
display(channel_df.style.format({
    "sessions": "{:,.0f}", "share_pct": "{:.1f}%", "view_pct": "{:.1f}%",
    "cart_from_view": "{:.1f}%", "purchase_pct": "{:.2f}%", "rev_per_session": "${:,.2f}",
    "view_item": "{:,.0f}", "add_to_cart": "{:,.0f}", "purchase": "{:,.0f}", "revenue": "${:,.0f}",
}))

site_view_rate = funnel_df.loc[funnel_df["step"] == "2. View Item", "overall_cvr"].iloc[0]
plot_df = channel_df.sort_values("view_pct")
fig_ch = go.Figure(go.Bar(
    y=plot_df["channel"], x=plot_df["view_pct"].round(1), orientation="h",
    text=[f"{v:.1f}%" for v in plot_df["view_pct"]], textposition="outside",
    marker_color=["#dc2626" if v < site_view_rate - 5 else "#4C78A8" for v in plot_df["view_pct"]],
))
fig_ch.add_vline(x=site_view_rate, line_dash="dash", line_color="#6b7280",
                 annotation_text=f"site avg {site_view_rate:.1f}%", annotation_position="top")
fig_ch.update_layout(title=f"Session → View Item Rate by Channel  ({channel_dim})",
                     height=max(300, 46 * len(plot_df) + 100), showlegend=False,
                     xaxis_title="% of sessions reaching a product page")
fig_ch.show()

# Decision rule: spread across actionable channels (excludes the 'other' bucket and '(not set)').
ranked = channel_df[~channel_df["channel"].str.startswith(("other (", "(not set)"))]
spread = ranked["view_pct"].max() - ranked["view_pct"].min()
best, worst = ranked.loc[ranked["view_pct"].idxmax()], ranked.loc[ranked["view_pct"].idxmin()]
print(f"View-item spread: {spread:.1f}pp  ({worst['channel']} {worst['view_pct']:.1f}% → {best['channel']} {best['view_pct']:.1f}%)")
# Within 3pp of the threshold the verdict is borderline, not settled.
if spread >= 10:
    verdict = "traffic mix is a driver — check landing pages / targeting for the weakest channel"
elif spread >= 7:
    verdict = (f"borderline — leaning 'the leak is on the site', but close to the threshold; "
               f"review {worst['channel']} landing pages before a navigation redesign")
else:
    verdict = "uniform across channels — the leak is on the site (navigation / discovery)"
print("Verdict:", verdict)
print("Note: the low_n flag applies to purchase_pct / revenue only — view_pct rests on session counts.")

# (not set) is excluded from the spread but is a large block — disclose it.
ns = channel_df[channel_df["channel"] == "(not set)"]
if len(ns):
    print(f"(not set): {ns['share_pct'].iloc[0]:.1f}% of sessions, view rate {ns['view_pct'].iloc[0]:.1f}% "
          f"(site average {site_view_rate:.1f}%)")

,channel,sessions,view_item,add_to_cart,purchase,revenue,share_pct,view_pct,cart_from_view,purchase_pct,rev_per_session,low_n
0,organic,"102,788","27,055","5,191",930,"$59,421",29.0%,26.3%,19.2%,0.90%,$0.58,False
1,(not set),"92,892","6,205",564,1,$12,26.2%,6.7%,9.1%,0.00%,$0.00,True
2,referral,"80,725","24,443","5,838","1,385","$103,656",22.7%,30.3%,23.9%,1.72%,$1.28,False
3,(none),"37,670","8,543","1,576",182,"$14,403",10.6%,22.7%,18.4%,0.48%,$0.38,False
4,,"25,302","5,608",954,66,"$5,381",7.1%,22.2%,17.0%,0.26%,$0.21,False
5,cpc,"7,664","1,653",268,28,"$2,009",2.2%,21.6%,16.2%,0.37%,$0.26,True
6,(data deleted),"6,315","1,455",401,109,"$7,042",1.8%,23.0%,27.6%,1.73%,$1.12,False
7,other (2 small channels),"1,501",299,78,20,"$1,754",0.4%,19.9%,26.1%,1.33%,$1.17,True


View-item spread: 8.7pp  (cpc 21.6% → referral 30.3%)
Verdict: borderline — leaning 'the leak is on the site', but close to the threshold; review cpc landing pages before a navigation redesign
Note: the low_n flag applies to purchase_pct / revenue only — view_pct rests on session counts.
(not set): 26.2% of sessions, view rate 6.7% (site average 21.2%)


## 6. Time-to-convert
For sessions that **complete the full funnel**, compute the median time (in hours)
between each consecutive step. This tells you how long the purchase consideration cycle is
and where the longest pauses happen.

In [11]:
# ── Section 6: Time-to-convert (BigQuery) ─────────────────────────────────
# For sessions that completed the full funnel, compute median hours between steps.
# GA4 event_timestamp is microseconds since epoch; divide by 3.6e12 for hours.
ttc_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT user_pseudo_id, session_id, event_name, MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END) AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END) AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END) AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END) AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END) AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),
-- full_funnel: only sessions with valid sequential timestamps through purchase.
full_funnel AS (
  SELECT *
  FROM pivoted
  WHERE
    ts_session  IS NOT NULL
    AND ts_view     >= ts_session
    AND ts_cart     >= ts_view
    AND ts_checkout >= ts_cart
    AND ts_purchase >= ts_checkout
)
SELECT
  -- Gaps in microseconds → convert to hours (p50 = median via APPROX_QUANTILES).
  ROUND(APPROX_QUANTILES((ts_view     - ts_session)  / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_session_to_view_hrs,
  ROUND(APPROX_QUANTILES((ts_cart     - ts_view)     / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_view_to_cart_hrs,
  ROUND(APPROX_QUANTILES((ts_checkout - ts_cart)     / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_cart_to_checkout_hrs,
  ROUND(APPROX_QUANTILES((ts_purchase - ts_checkout) / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_checkout_to_purchase_hrs,
  ROUND(APPROX_QUANTILES((ts_purchase - ts_session)  / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_total_hrs,
  COUNT(*) AS converting_sessions
FROM full_funnel
"""

ttc_df = client.query(ttc_sql).to_dataframe()
print("Median time between funnel steps (hours) — converting sessions only:")
display(ttc_df.T.rename(columns={0: "median_hours"}))


Median time between funnel steps (hours) — converting sessions only:


,median_hours
p50_session_to_view_hrs,0.02
p50_view_to_cart_hrs,0.0
p50_cart_to_checkout_hrs,0.11
p50_checkout_to_purchase_hrs,0.06
p50_total_hrs,0.26
converting_sessions,2721


In [12]:
# ── Section 6b: Time-to-convert bar chart ─────────────────────────────────
# Reshape median hour columns into a simple table for Plotly.
ttc_steps = pd.DataFrame({
    "transition": [
        "Session → View Item",
        "View Item → Add to Cart",
        "Add to Cart → Checkout",
        "Checkout → Purchase",
    ],
    "median_hours": [
        float(ttc_df["p50_session_to_view_hrs"].iloc[0]),
        float(ttc_df["p50_view_to_cart_hrs"].iloc[0]),
        float(ttc_df["p50_cart_to_checkout_hrs"].iloc[0]),
        float(ttc_df["p50_checkout_to_purchase_hrs"].iloc[0]),
    ],
})

fig_ttc = px.bar(
    ttc_steps, x="transition", y="median_hours",
    title="Median Time Between Funnel Steps (Converting Sessions)",
    labels={"median_hours": "Median hours", "transition": ""},
    color="median_hours",
    color_continuous_scale="Blues",
    text="median_hours",
)
fig_ttc.update_traces(texttemplate="%{text:.2f}h", textposition="outside")
fig_ttc.update_layout(height=400, showlegend=False, coloraxis_showscale=False)
fig_ttc.show()


## 7. Shoppers funnel — alternative entry at `view_item`
Starting at `view_item` removes sessions that bounced before ever seeing a product.
This separates **shoppers** (people with product intent) from all traffic,
and gives the merchandising/PDP team a cleaner conversion rate to own.

**Hypothesis:** if this funnel shows a materially higher `add_to_cart` rate than the
full funnel, the main problem is getting people to products — not the product page itself.

In [13]:
# ── Section 7: Shoppers funnel — entry at view_item (BigQuery) ────────────
# Alternative funnel: only sessions where the user viewed a product.
# Removes "bounced" sessions that never reached a PDP — isolates shopping intent.
shoppers_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT user_pseudo_id, session_id, event_name, MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END) AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END) AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END) AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END) AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
)
SELECT
  COUNT(*)                                                                        AS step1_view_item,
  COUNTIF(ts_cart     IS NOT NULL AND ts_cart     >= ts_view)                    AS step2_add_to_cart,
  COUNTIF(ts_checkout IS NOT NULL AND ts_checkout >= ts_cart
                                 AND ts_cart      >= ts_view)                    AS step3_begin_checkout,
  COUNTIF(ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout
                                 AND ts_checkout  >= ts_cart
                                 AND ts_cart      >= ts_view)                    AS step4_purchase
FROM pivoted
WHERE ts_view IS NOT NULL
"""

shoppers_raw = client.query(shoppers_sql).to_dataframe()
print("Shoppers funnel (entry at view_item):")
display(shoppers_raw)


Shoppers funnel (entry at view_item):


,step1_view_item,step2_add_to_cart,step3_begin_checkout,step4_purchase
0,77020,15146,5293,2751


In [14]:
# ── Section 7b: Shoppers funnel metrics + comparison ──────────────────────
shopper_labels = ["1. View Item", "2. Add to Cart", "3. Begin Checkout", "4. Purchase"]
shopper_cols   = ["step1_view_item","step2_add_to_cart","step3_begin_checkout","step4_purchase"]

shoppers_df = pd.DataFrame({
    "step":     shopper_labels,
    "sessions": [int(shoppers_raw[c].iloc[0]) for c in shopper_cols],
})
shoppers_df["step_cvr"]    = (shoppers_df["sessions"] / shoppers_df["sessions"].shift(1) * 100).round(1)
shoppers_df["overall_cvr"] = (shoppers_df["sessions"] / shoppers_df["sessions"].iloc[0] * 100).round(1)
shoppers_df["drop_off"]    = (100 - shoppers_df["step_cvr"]).round(1)

print("Shoppers funnel conversion rates:")
display(shoppers_df)

fig_shop = px.funnel(
    shoppers_df, x="sessions", y="step",
    title="Shoppers Funnel (Entry: View Item)",
    color_discrete_sequence=["#54A24B"],
)
for i, row in shoppers_df.iterrows():
    if i == 0:
        continue
    fig_shop.add_annotation(
        x=row["sessions"], y=row["step"],
        text=f"  ↓ {row['drop_off']}% drop-off  ({row['step_cvr']}% pass)",
        showarrow=False, xanchor="left",
        font=dict(size=11, color="#d62728"),
    )
fig_shop.update_layout(height=400, margin=dict(l=160, r=260, t=60, b=40))
fig_shop.show()

# Compare add-to-cart from session start vs from product viewers (diagnostic split).
full_cart_rate    = funnel_df.loc[funnel_df["step"]=="3. Add to Cart", "overall_cvr"].values[0]
shopper_cart_rate = shoppers_df.loc[shoppers_df["step"]=="2. Add to Cart", "step_cvr"].values[0]
print(f"\nAdd-to-cart rate from session start:  {full_cart_rate:.1f}%")
print(f"Add-to-cart rate from view_item:       {shopper_cart_rate:.1f}%")
print(f"\nIf these are similar → the PDP is the bottleneck.")
print(f"If the shoppers rate is much higher → the problem is getting users to products.")


Shoppers funnel conversion rates:


,step,sessions,step_cvr,overall_cvr,drop_off
0,1. View Item,77020,NaN,100.0,NaN
1,2. Add to Cart,15146,19.7,19.7,80.3
2,3. Begin Checkout,5293,34.9,6.9,65.1
3,4. Purchase,2751,52.0,3.6,48.0



Add-to-cart rate from session start:  4.2%
Add-to-cart rate from view_item:       19.7%

If these are similar → the PDP is the bottleneck.
If the shoppers rate is much higher → the problem is getting users to products.


## 7c. Product category — where view → add-to-cart breaks down

The shoppers funnel shows *how much* product-page interest fails to become intent.
This section shows *where*: the view → add-to-cart rate for each `item_category`,
so the product-page audit recommended in the brief starts from a ranked list
rather than a guess.

**Grain:** one row = one (session, item_category) pair, built from `UNNEST(items)`.
A session counts as *carting* a category only if it added an item **of that
category** at or after first viewing that category in the same session.

**Why a separate query, not another funnel split?** The `items` array exists only
on `view_item`, `add_to_cart` and `purchase` — `session_start` and `begin_checkout`
carry no category — so category cannot be a dimension of the session funnel CTE.

**Label normalisation.** `item_category` mixes flat labels (`Bags`) with paths
(`Lifestyle/Bags/`, `Home/Apparel/`). Ranked raw, the same category appears several
times with its volume split. Labels are reduced to their last path segment in SQL —
before the per-session grouping, so a session that viewed both labels counts once —
and the merges are listed for review.

**Reading the numbers**
- Rates are stricter than the shoppers ATC rate (adding a *different* category's item doesn't count).
- Rows are not additive: one session can view several categories.
- Ranking uses **carts at stake** = (benchmark rate − category rate) × viewing sessions,
  so a high-traffic category with a moderate gap outranks a niche one with a dramatic rate.

In [15]:
# ── Section 7c: Product category view → cart (BigQuery) ──────────────────────
# One row per (session, item_category). MAX(cart) >= MIN(view) means "at least
# one add-to-cart of this category at or after the first view of it".
category_sql = f"""
-- item_events: one row per item on each view_item / add_to_cart event.
WITH item_events AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1)                     AS session_id,
    event_name,
    event_timestamp,
    COALESCE(NULLIF(TRIM(it.item_category), ''), '(not set)')    AS raw_category
  FROM `{DATASET}.events_*`, UNNEST(items) AS it
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('view_item', 'add_to_cart')
),

-- normalised: item_category mixes flat labels ("Bags") with paths ("Lifestyle/Bags/").
-- Keep the last path segment. Done BEFORE the per-session grouping so a session that
-- viewed both "Bags" and "Lifestyle/Bags/" counts once, not twice.
-- ' / ' (slash with spaces) is part of a name, e.g. "Men's / Unisex", not a path separator.
normalised AS (
  SELECT
    *,
    IF(raw_category = '(not set)', raw_category,
       COALESCE(NULLIF(TRIM(REPLACE(ARRAY_REVERSE(SPLIT(TRIM(REPLACE(raw_category, ' / ', ' | '), '/'), '/'))[SAFE_OFFSET(0)], ' | ', ' / ')), ''),
                '(not set)'))                                     AS item_category
  FROM item_events
),

-- labels: audit trail of which raw labels were merged into each category.
labels AS (
  SELECT
    item_category,
    COUNT(DISTINCT raw_category)                                    AS n_source_labels,
    STRING_AGG(DISTINCT raw_category, ' · ' ORDER BY raw_category)  AS source_labels
  FROM normalised
  GROUP BY item_category
),

-- session_category: first view and last add-to-cart per (session, category).
session_category AS (
  SELECT
    user_pseudo_id, session_id, item_category,
    MIN(IF(event_name = 'view_item',   event_timestamp, NULL))   AS ts_first_view,
    MAX(IF(event_name = 'add_to_cart', event_timestamp, NULL))   AS ts_last_cart
  FROM normalised
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, item_category
),

agg AS (
  SELECT
    item_category,
    COUNTIF(ts_first_view IS NOT NULL)                                  AS viewing_sessions,
    COUNTIF(ts_first_view IS NOT NULL AND ts_last_cart >= ts_first_view) AS carting_sessions
  FROM session_category
  GROUP BY item_category
)

SELECT agg.*, labels.n_source_labels, labels.source_labels
FROM agg
JOIN labels USING (item_category)
ORDER BY viewing_sessions DESC
"""

category_raw = client.query(category_sql).to_dataframe()

# Data-quality check before ranking anything: how much of the catalogue is untagged?
_views = category_raw["viewing_sessions"].sum()
_notset = category_raw.loc[category_raw["item_category"] == "(not set)", "viewing_sessions"].sum()
print(f"Raw labels: {int(category_raw['n_source_labels'].sum())}  →  categories after normalising: {len(category_raw)}")
print(f"'(not set)' share of category views: {_notset/_views*100:.1f}%")
print("Merged categories — check none of these merges two genuinely different things:")
display(category_raw[category_raw["n_source_labels"] > 1][["item_category", "n_source_labels", "source_labels", "viewing_sessions"]])

Raw labels: 74  →  categories after normalising: 47
'(not set)' share of category views: 5.0%
Merged categories — check none of these merges two genuinely different things:


,item_category,n_source_labels,source_labels,viewing_sessions
0,Men's / Unisex,2,Home/Apparel/Men's / Unisex/ · Men's / Unisex,13672
1,Sale,2,Home/Sale/ · Sale,9763
2,Apparel,2,Apparel · Home/Apparel/,6548
3,Hats,2,Hats · Home/Apparel/Hats/,6226
4,Drinkware,2,Drinkware · Lifestyle/Drinkware/,5382
5,New,2,Home/New/ · New,5335
7,Bags,2,Bags · Lifestyle/Bags/,4796
8,Women's,2,Home/Apparel/Women's/ · Women's,4778
9,YouTube,2,Home/Shop by Brand/YouTube/ · YouTube,4754
10,Small Goods,2,Lifestyle/Small Goods/ · Small Goods,3951


In [16]:
# ── Section 7d: Category ranking + chart ─────────────────────────────────────
MIN_CATEGORY_VIEWS = 500   # below this the rate is too noisy to rank

category_df = category_raw[
    (category_raw["item_category"] != "(not set)")
    & (category_raw["viewing_sessions"] >= MIN_CATEGORY_VIEWS)
].copy()
category_df["atc_pct"] = category_df["carting_sessions"] / category_df["viewing_sessions"] * 100

# Benchmark: pooled (volume-weighted) rate across ranked categories.
category_benchmark = category_df["carting_sessions"].sum() / category_df["viewing_sessions"].sum() * 100

# Carts at stake: extra carting sessions if the category converted at the benchmark.
category_df["carts_at_stake"] = (
    (category_benchmark - category_df["atc_pct"]).clip(lower=0) / 100
    * category_df["viewing_sessions"]
).round().astype(int)
category_df = category_df.sort_values("atc_pct").reset_index(drop=True)

print(f"Ranked categories: {len(category_df)}  |  benchmark view → cart: {category_benchmark:.1f}%")
display(category_df.sort_values("carts_at_stake", ascending=False)
        .drop(columns=["source_labels"])
        .style.format({"atc_pct": "{:.1f}%", "viewing_sessions": "{:,}",
                       "carting_sessions": "{:,}", "carts_at_stake": "{:,}"}))

# Worst rate at the top; below-benchmark categories highlighted.
fig_cat = go.Figure(go.Bar(
    y=category_df["item_category"], x=category_df["atc_pct"].round(1),
    orientation="h",
    text=[f"{v:.1f}%" for v in category_df["atc_pct"]], textposition="outside",
    marker_color=["#ea580c" if v < category_benchmark else "#4C78A8" for v in category_df["atc_pct"]],
))
fig_cat.add_vline(x=category_benchmark, line_dash="dash", line_color="#6b7280",
                  annotation_text=f"benchmark {category_benchmark:.1f}%", annotation_position="top")
fig_cat.update_layout(
    title="View → Add-to-Cart Rate by Product Category",
    height=max(320, 30 * len(category_df) + 100),
    yaxis=dict(autorange="reversed"),
    xaxis_title="% of viewing sessions that add that category to cart",
    showlegend=False,
)
fig_cat.show()

top_targets = (category_df.sort_values("carts_at_stake", ascending=False)
               .query("carts_at_stake > 0").head(3))
print("Start the product-page audit with:",
      ", ".join(f"{r.item_category} ({r.carts_at_stake:,} carts at stake)" for r in top_targets.itertuples()))

Ranked categories: 23  |  benchmark view → cart: 25.0%


,item_category,viewing_sessions,carting_sessions,n_source_labels,atc_pct,carts_at_stake
0,YouTube,"4,754",744,2,15.6%,444
3,Apparel,"6,548","1,252",2,19.1%,385
2,Bags,"4,796",843,2,17.6%,356
4,New,"5,335","1,030",2,19.3%,303
5,Eco-Friendly,"3,725",740,2,19.9%,191
1,Men's T-Shirts,"1,810",291,2,16.1%,161
6,Lifestyle,"2,755",585,2,21.2%,104
10,Small Goods,"3,951",923,2,23.4%,64
9,Google,"3,234",747,2,23.1%,61
12,Men's / Unisex,"13,672","3,378",2,24.7%,39


Start the product-page audit with: YouTube (444 carts at stake), Apparel (385 carts at stake), Bags (356 carts at stake)


## 8. Weekly cohort retention
Groups users by the week of their first `session_start`, then tracks what fraction
return in each subsequent week across the full Nov 2020–Jan 2021 window.

Two boundary effects are handled before any average is taken:

- **Right-censoring.** A cohort's week N is only observable if that whole week falls
  inside the data window. Unobservable cells are left **blank, not 0%** — a user who
  joined in the final week has not had the chance to return yet.
- **Partial first cohort.** Cohorts are Monday-anchored but the window opens on a
  Sunday, so the first bucket holds a single day of users. Its small base makes the
  rate noisy, so it is **excluded from the averages**.

In [17]:
# ── Section 8: Weekly cohort retention (BigQuery) ───────────────────────────
# Cohort = calendar week (Monday) of each user's first session_start.
# Retention = distinct users with session_start in week N after cohort week.
cohort_sql = f"""
WITH first_sessions AS (
  SELECT
    user_pseudo_id,
    DATE_TRUNC(PARSE_DATE('%Y%m%d', MIN(event_date)), WEEK(MONDAY)) AS cohort_week
  FROM `{DATASET}.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name = 'session_start'
  GROUP BY user_pseudo_id
),
activity AS (
  SELECT DISTINCT
    user_pseudo_id,
    DATE_TRUNC(PARSE_DATE('%Y%m%d', event_date), WEEK(MONDAY)) AS activity_week
  FROM `{DATASET}.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name = 'session_start'
)
SELECT
  f.cohort_week,
  DATE_DIFF(a.activity_week, f.cohort_week, WEEK) AS weeks_since_cohort,
  COUNT(DISTINCT a.user_pseudo_id)                 AS active_users
FROM first_sessions f
JOIN activity a USING (user_pseudo_id)
WHERE a.activity_week >= f.cohort_week
GROUP BY f.cohort_week, weeks_since_cohort
ORDER BY f.cohort_week, weeks_since_cohort
"""

cohort_df = client.query(cohort_sql).to_dataframe()
print(f"Cohort data rows: {len(cohort_df)}")
display(cohort_df.head(15))


Cohort data rows: 105


,cohort_week,weeks_since_cohort,active_users
0,2020-10-26,0,2339
1,2020-10-26,1,251
2,2020-10-26,2,82
3,2020-10-26,3,56
4,2020-10-26,4,31
5,2020-10-26,5,24
6,2020-10-26,6,22
7,2020-10-26,7,5
8,2020-10-26,8,14
9,2020-10-26,9,5


In [18]:
# ── Section 8b: Retention heatmap — censoring + partial-cohort guards ─────
# Same logic as generate_dashboard.py, so notebook and dashboard agree.
import datetime as _dt

cohort_pivot = cohort_df.pivot(index="cohort_week", columns="weeks_since_cohort",
                               values="active_users")

# Censoring guard: week N of a cohort is observed only if the whole week fits in
# the window. Unobserved → NaN (blank); observed but nobody returned → real 0.
_start = _dt.date(int(DATE_START[:4]), int(DATE_START[4:6]), int(DATE_START[6:8]))
_end   = _dt.date(int(DATE_END[:4]),   int(DATE_END[4:6]),   int(DATE_END[6:8]))
_cohort_dates = [pd.Timestamp(d).date() for d in cohort_pivot.index]
_observed = pd.DataFrame(
    [[(cd + _dt.timedelta(days=7 * int(k) + 6)) <= _end for k in cohort_pivot.columns]
     for cd in _cohort_dates],
    index=cohort_pivot.index, columns=cohort_pivot.columns,
)
cohort_pivot = cohort_pivot.where(_observed)
cohort_pivot = cohort_pivot.mask(_observed & cohort_pivot.isna(), 0)
censored = int((~_observed).sum().sum())
print(f"Censored cells (blank, not 0%): {censored} of {_observed.size} "
      f"({censored / _observed.size * 100:.0f}%)")

cohort_pct = cohort_pivot.div(cohort_pivot[0], axis=0).multiply(100).round(1)

# Partial-first-cohort guard: Monday-anchored buckets + a window that opens
# mid-week → the first bucket is a partial week with a much smaller base.
partial_first = _cohort_dates[0] < _start
if partial_first:
    print(f"Partial first cohort {_cohort_dates[0]}: {int(cohort_pivot[0].iloc[0]):,} users "
          f"vs {int(cohort_pivot[0].iloc[1]):,} in the first full cohort — excluded from averages")

cohort_pct.index   = cohort_pct.index.astype(str)
cohort_pct.columns = [f"Week {c}" for c in cohort_pct.columns]
cohort_pct_full    = cohort_pct.iloc[1:] if partial_first else cohort_pct

print("Retention matrix (% of cohort returning; blank = not yet observable):")
display(cohort_pct)

z   = cohort_pct.values.tolist()
txt = [["" if pd.isna(v) else f"{v:.0f}%" for v in row] for row in z]

# Blues runs light → dark as the value rises, so darker = better retention.
# zmax=12 keeps the 0–12% band readable; Week 0 is 100% by definition.
fig_ret = go.Figure(go.Heatmap(
    z=z, x=cohort_pct.columns.tolist(), y=cohort_pct.index.tolist(),
    text=txt, texttemplate="%{text}",
    colorscale="Blues", zmin=0, zmax=12, hoverongaps=False,
))
fig_ret.update_layout(
    title="Weekly Cohort Retention (% returning)",
    xaxis_title="Weeks Since First Session",
    yaxis_title="Cohort Week",
    height=500,
    yaxis_autorange="reversed",
)
fig_ret.show()

# Headline KPIs: full cohorts only; censored (NaN) cells are skipped by .mean().
if "Week 1" in cohort_pct_full.columns:
    print(f"\nAvg Week-1 retention (full cohorts): {cohort_pct_full['Week 1'].mean():.1f}%")
if "Week 2" in cohort_pct_full.columns:
    print(f"Avg Week-2 retention (full cohorts): {cohort_pct_full['Week 2'].mean():.1f}%")

Censored cells (blank, not 0%): 91 of 196 (46%)
Partial first cohort 2020-10-26: 2,339 users vs 19,194 in the first full cohort — excluded from averages
Retention matrix (% of cohort returning; blank = not yet observable):


,Week 0,Week 1,Week 2,Week 3,Week 4,Week 5,Week 6,Week 7,Week 8,Week 9,Week 10,Week 11,Week 12,Week 13
cohort_week,,,,,,,,,,,,,,
2020-10-26,100.0,10.7,3.5,2.4,1.3,1.0,0.9,0.2,0.6,0.2,0.4,0.3,0.3,0.3
2020-11-02,100.0,6.2,3.4,2.3,2.3,1.9,1.3,0.6,0.4,0.7,0.5,0.4,0.4,<NA>
2020-11-09,100.0,6.7,3.0,2.6,2.4,1.5,0.8,0.4,0.6,0.5,0.5,0.4,<NA>,<NA>
2020-11-16,100.0,5.7,3.3,2.4,1.8,0.7,0.4,0.5,0.5,0.4,0.3,<NA>,<NA>,<NA>
2020-11-23,100.0,5.0,2.5,1.5,0.7,0.4,0.5,0.5,0.4,0.3,<NA>,<NA>,<NA>,<NA>
2020-11-30,100.0,5.3,2.2,0.9,0.6,0.7,0.5,0.4,0.4,<NA>,<NA>,<NA>,<NA>,<NA>
2020-12-07,100.0,3.3,0.9,0.7,1.0,0.6,0.4,0.4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2020-12-14,100.0,2.9,0.9,1.2,0.9,0.6,0.4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2020-12-21,100.0,2.5,1.1,1.0,0.6,0.4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>



Avg Week-1 retention (full cohorts): 4.2%
Avg Week-2 retention (full cohorts): 1.9%


## 9. Summary & hypotheses

| Metric | What to look for | Hypothesis if worse than expected |
|---|---|---|
| **Session → View Item** | Should be the biggest absolute drop | Landing page / traffic quality; weak homepage routing to products |
| **View Item → Add to Cart** | Second-biggest lever | PDP UX, pricing, stock, missing trust signals |
| **Add to Cart → Purchase** | Usually high (>50%) for e-comm | Checkout friction, shipping cost surprise, payment issues |
| **Shoppers funnel ATC rate** | Compare to full-funnel ATC | If similar → PDP is bottleneck; if much higher → navigation/discovery is bottleneck |
| **AOV / median order value** | Baseline for revenue modelling | Segment by device or traffic source to find high-value cohorts |
| **Time-to-convert** | Checkout → Purchase should be <5 min | If long, investigate payment flow or shipping calculator UX |
| **Week-1 retention** | Benchmark: ~20%+ is healthy for e-comm | No email/push, weak remarketing, one-and-done campaigns |
| **Mobile vs desktop** | Expect mobile to lag at checkout | If similar, focus on merchandising & product discovery, not checkout |

---

### Next steps

**Point at real data:** swap `bigquery-public-data.ga4_obfuscated_sample_ecommerce` for your own
GA4 BigQuery export dataset. The SQL patterns are identical.

**Add segments:** channel (Section 5c) and product category (Section 7c) are done.
Next candidates: `geo.country`, or a `new_vs_returning` flag (`ga_session_number = 1` = new user).
When segmenting a *session* funnel, use session-level attributes — `traffic_source.*`
is first-touch and user-level (see the grain note in 5c).

**Statistical rigour:** for device or period comparisons, add a proportion Z-test
to confirm differences are real and not sampling noise — especially important
when tablet N is small.

## 10. AI-Powered Insight Generation

Pass all analysis results to an LLM to produce a **product-decision-ready brief** —
the kind of output a Product Analyst shares with a squad after completing an analysis.

This section demonstrates three things that map directly to modern product analytics roles:

- **Prompt engineering** — a structured system prompt with an explicit output schema
- **Contextualised LLM calls** — serialising DataFrame results as structured JSON context
- **AI-assisted insight workflows** — replacing manual write-up time with a reproducible, auditable step

> **API key (optional).** Keys are read from environment variables only — never paste
> one into the notebook, because outputs are committed.
> - **Anthropic Claude** (paid): key from [console.anthropic.com](https://console.anthropic.com) → `export ANTHROPIC_API_KEY=...` → used by **10d**
> - **Google Gemini** (free): key from [aistudio.google.com](https://aistudio.google.com) → `export GEMINI_API_KEY=...` → used by **10e**, only if 10d produced no brief
> - **Neither set:** 10d–10f are skipped. The committed `dashboard.html` keeps the brief it already embeds.
>
> 10f writes `product_brief_ga4_funnel.md`. It is generated output and gitignored;
> `generate_dashboard.py` embeds it when present.

In [19]:
# ── Section 10a: Install SDK(s) ─────────────────────────────────────────────
# Install the Anthropic SDK (used in 10d). If you prefer the free Gemini
# alternative, also install google-genai (see Section 10e).
%pip install -q anthropic
%pip install -q google-genai


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [20]:
# ── Section 10b: API key ───────────────────────────────────────────────────
import os

# Recommended: set in your shell before launching Jupyter
#   export ANTHROPIC_API_KEY="sk-ant-..."
# Fallback: paste key directly here for local use only (never commit).
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")

if ANTHROPIC_API_KEY == "YOUR_API_KEY_HERE":
    print("Set ANTHROPIC_API_KEY before running Section 10.")
else:
    print("API key loaded.")


Set ANTHROPIC_API_KEY before running Section 10.


In [21]:
# ── Section 10c: Compile analysis context for the LLM ─────────────────────
import json

def _f(val):
    """Safely convert a pandas/numpy scalar to a rounded Python float (NaN → None)."""
    try:
        v = float(val)
        return None if v != v else round(v, 2)  # NaN → None
    except (TypeError, ValueError):
        return None

# ctx: structured dict passed as JSON in the user prompt to Claude.
ctx = {}

# Session funnel (Section 3) — requires funnel_df from prior cells.
try:
    ctx["session_funnel"] = [
        {
            "step": r["step"],
            "sessions": int(r["sessions"]),
            "step_conversion_pct": _f(r["step_cvr"]),
            "overall_conversion_pct": _f(r["overall_cvr"]),
            "drop_off_pct": _f(r["drop_off"]),
        }
        for _, r in funnel_df.iterrows()
    ]
except NameError:
    print("funnel_df not found — run Section 3 first")

# Revenue summary (Section 4).
try:
    rv = revenue_df.iloc[0]
    ctx["revenue"] = {
        "purchasing_sessions": int(rv["purchasing_sessions"]),
        "total_revenue_usd": _f(rv["total_revenue"]),
        "avg_order_value_usd": _f(rv["avg_order_value"]),
        "median_order_value_usd": _f(rv["median_order_value"]),
    }
except NameError:
    print("revenue_df not found — run Section 4 first")

# Device breakdown at purchase step (Section 5).
try:
    purchase_rows = device_long[device_long["step_col"] == "step5_purchase"]
    ctx["purchase_rate_by_device"] = [
        {
            "device": r["device_category"],
            "sessions": int(r["sessions"]),
            "pct_of_device_sessions": _f(r["pct"]),
        }
        for _, r in purchase_rows.iterrows()
    ]
except NameError:
    print("device_long not found — run Section 5 first")

# Median hours between funnel steps (Section 6).
try:
    ctx["median_time_between_steps_hours"] = {
        "session_to_view_item": _f(ttc_df["p50_session_to_view_hrs"].iloc[0]),
        "view_item_to_add_to_cart": _f(ttc_df["p50_view_to_cart_hrs"].iloc[0]),
        "add_to_cart_to_checkout": _f(ttc_df["p50_cart_to_checkout_hrs"].iloc[0]),
        "checkout_to_purchase": _f(ttc_df["p50_checkout_to_purchase_hrs"].iloc[0]),
        "total_session_to_purchase": _f(ttc_df["p50_total_hrs"].iloc[0]),
    }
except NameError:
    print("ttc_df not found — run Section 6 first")

# Shoppers vs full funnel add-to-cart comparison (Section 7).
try:
    full_cart = funnel_df.loc[funnel_df["step"] == "3. Add to Cart", "overall_cvr"].values
    shop_cart = shoppers_df.loc[shoppers_df["step"] == "2. Add to Cart", "step_cvr"].values
    ctx["shoppers_funnel"] = {
        "note": "Funnel starting at view_item — removes sessions that never saw a product",
        "add_to_cart_rate_from_all_sessions_pct": _f(full_cart[0]) if len(full_cart) else None,
        "add_to_cart_rate_from_product_viewers_pct": _f(shop_cart[0]) if len(shop_cart) else None,
    }
except NameError:
    print("shoppers_df not found — run Section 7 first")

# Cohort retention averages (Section 8).
try:
    # Full cohorts only (partial first cohort excluded); censored cells are NaN and skipped.
    wk1 = cohort_pct_full["Week 1"].mean() if "Week 1" in cohort_pct_full.columns else None
    wk2 = cohort_pct_full["Week 2"].mean() if "Week 2" in cohort_pct_full.columns else None
    ctx["cohort_retention"] = {
        "avg_week1_return_pct": _f(wk1),
        "avg_week2_return_pct": _f(wk2),
        "note": ("% of users with any session in week N after their acquisition week; "
                 "full cohorts only, right-censored weeks excluded. Measures return visits, "
                 "not purchases: it cannot show whether buyers decided in one session or "
                 "researched across sessions (cross-device and cleared cookies also look like "
                 "non-returns)."),
    }
except NameError:
    print("cohort_pct_full not found — run Section 8 first")

# Acquisition channel split (Section 5c).
try:
    ctx["funnel_by_channel"] = {
        "channel_definition": channel_dim,
        "view_item_spread_pp": _f(spread),
        "decision_rule": "10pp spread in view_item_pct = traffic-quality problem; 7-10pp = borderline; below 7pp = site problem",
        "verdict": verdict,
        "note": ("low_purchase_volume=True means fewer than 30 purchases: purchase_pct and "
                 "revenue_per_session are too noisy to rank or act on. view_item_pct rests on "
                 "session counts and is reliable. '(not set)' sessions carry no session-level "
                 "medium and are excluded from the spread."),
        "rows": [
            {
                "channel": r["channel"],
                "sessions": int(r["sessions"]),
                "view_item_pct": _f(r["view_pct"]),
                "purchase_pct": _f(r["purchase_pct"]),
                "revenue_per_session_usd": _f(r["rev_per_session"]),
                "low_purchase_volume": bool(r["low_n"]),
            }
            for _, r in channel_df.iterrows()
        ],
    }
except NameError:
    print("channel_df not found — run Section 5c first")

# Product category view → cart (Section 7c).
try:
    ctx["category_view_to_cart"] = {
        "benchmark_pct": _f(category_benchmark),
        "metric_definition": ("view_to_cart_pct = of sessions that viewed a product page (view_item) "
                              "in this category, the % that added an item from this category to the cart. "
                              "It starts AFTER the product page is reached, so it says nothing about "
                              "category/listing pages or clicks from them to product pages."),
        "note": "carts_at_stake = extra carting sessions if the category converted at the benchmark",
        "rows": [
            {
                "category": r["item_category"],
                "viewing_sessions": int(r["viewing_sessions"]),
                "view_to_cart_pct": _f(r["atc_pct"]),
                "carts_at_stake": int(r["carts_at_stake"]),
            }
            for _, r in category_df.sort_values("carts_at_stake", ascending=False).iterrows()
        ],
    }
except NameError:
    print("category_df not found — run Section 7c first")

print("Context compiled — passing to LLM:")
# default=str avoids crashes if any value is still a numpy/pandas scalar
print(json.dumps(ctx, indent=2, default=str))


Context compiled — passing to LLM:
{
  "session_funnel": [
    {
      "step": "1. Session Start",
      "sessions": 354857,
      "step_conversion_pct": null,
      "overall_conversion_pct": 100.0,
      "drop_off_pct": null
    },
    {
      "step": "2. View Item",
      "sessions": 75261,
      "step_conversion_pct": 21.2,
      "overall_conversion_pct": 21.2,
      "drop_off_pct": 78.8
    },
    {
      "step": "3. Add to Cart",
      "sessions": 14870,
      "step_conversion_pct": 19.8,
      "overall_conversion_pct": 4.2,
      "drop_off_pct": 80.2
    },
    {
      "step": "4. Begin Checkout",
      "sessions": 5205,
      "step_conversion_pct": 35.0,
      "overall_conversion_pct": 1.5,
      "drop_off_pct": 65.0
    },
    {
      "step": "5. Purchase",
      "sessions": 2721,
      "step_conversion_pct": 52.3,
      "overall_conversion_pct": 0.8,
      "drop_off_pct": 47.7
    }
  ],
  "revenue": {
    "purchasing_sessions": 2461,
    "total_revenue_usd": 193678.0,
    "av

In [22]:
# ── Section 10d: Generate product brief via Claude ─────────────────────────
import anthropic

# System prompt: fixed output schema + product-analyst tone (not data-science jargon).
SYSTEM_PROMPT = """You are a senior Product Analyst embedded in a cross-functional product squad.
Your job is to translate funnel analysis results into clear, decision-ready product briefs.

Always produce exactly these five sections:

## Executive Summary
2-3 sentences. State the single most important finding and its business implication.

## Key Findings
3-5 numbered findings. Each must include a specific number from the data and a
"so what" that a product manager can act on.

## Hypotheses
3 prioritised hypotheses that could explain the patterns.
Label each: [HIGH / MEDIUM / LOW] confidence based on how strongly the data supports it.

## Recommended Actions
3 specific, testable actions. For each state: what to do, why, and what metric to watch.

## What to Measure Next
2-3 analyses that would either confirm these hypotheses or reveal what this analysis misses.

Rules:
- Be opinionated. Choose a priority — do not hedge everything equally.
- Write for a product manager, not a data scientist. No statistical jargon.
- Flag explicitly where findings come from a public sample and need validation on real data.
- Respect the data-quality flags and verdicts in the data. Never base a finding or an
  action on a metric flagged as low volume.
- The session → view_item drop is not a bounce rate; do not call it one.
- Use each metric only for what its definition in the data says it measures. A
  product-page-to-cart rate is evidence about product pages (price, imagery, sizing,
  stock, description), not about category or listing pages.
- Low return-visit rates do not show that purchases are impulse or single-session
  decisions; do not claim buyer motivation the data cannot show.
- Where relevant, note how e-commerce metrics translate to B2B SaaS equivalents
  (e.g. session_start → login, view_item → feature discovery, purchase → activation)."""

# User prompt embeds compiled ctx JSON from the previous cell.
USER_PROMPT = f"""Produce a product brief from these funnel analysis results.

Dataset: GA4 public e-commerce sample (Nov 2020 – Jan 2021, session-scoped funnel).
The methodology is directly transferable to B2B SaaS — where "purchase" maps to
a key activation event such as completing onboarding or adopting a core feature.

The acquisition-channel split (funnel_by_channel) and the product-category split
(category_view_to_cart) have ALREADY been done — their results are in the data below.
Interpret them; do not recommend running a traffic-source / channel or category
segmentation as a next step.

{json.dumps(ctx, indent=2)}"""


def generate_brief(api_key: str) -> str:
    """Call Claude with system + user prompts; return markdown brief text."""
    client = anthropic.Anthropic(api_key=api_key)
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1800,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": USER_PROMPT}],
    )
    return response.content[0].text


if ANTHROPIC_API_KEY != "YOUR_API_KEY_HERE":
    print("Calling Claude... (~10-20 seconds)\n")
    brief = generate_brief(ANTHROPIC_API_KEY)
    BRIEF_MODEL = "Claude (claude-sonnet-4-6)"
    print(brief)
else:
    brief = None
    print("Skipped — set ANTHROPIC_API_KEY first.")


Skipped — set ANTHROPIC_API_KEY first.


In [23]:
# ── Section 10e: FREE ALTERNATIVE — Google Gemini ───────────────────────────
# Runs only if 10d did not produce a brief (no Anthropic key) and a Gemini key is set.
# Setup (one-time): get a free key at https://aistudio.google.com → "Get API key",
#   export GEMINI_API_KEY="..."   (never paste the key into the notebook — outputs get committed)
# Uses the google-genai SDK; the older google-generativeai package is deprecated.
# Model IDs change often — check https://ai.google.dev/gemini-api/docs/models and
# override with export GEMINI_MODEL="..." if the default is retired.
import os

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_KEY_HERE")
GEMINI_MODEL   = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash")

if brief:
    print("Brief already generated in 10d — skipping Gemini.")
elif GEMINI_API_KEY == "YOUR_GEMINI_KEY_HERE":
    print("Skipped — set GEMINI_API_KEY (free) or ANTHROPIC_API_KEY to generate a brief.")
else:
    from google import genai
    from google.genai import types

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)   # not `client`: that is BigQuery
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=USER_PROMPT,                               # from Section 10d
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,               # from Section 10d
            max_output_tokens=8192,   # thinking tokens count toward this on Flash models
        ),
    )
    brief = response.text
    BRIEF_MODEL = f"Gemini ({GEMINI_MODEL})"
    _finish = response.candidates[0].finish_reason if response.candidates else None
    if str(_finish).endswith("MAX_TOKENS"):
        print("WARNING: brief was cut off at max_output_tokens — raise it and re-run.\n")
    print(brief)

## Executive Summary
The primary growth bottleneck on the platform is a drop-off prior to intent: 78.8% of sessions end before a user reaches a single product detail page (PDP), and product pages in high-volume categories like YouTube merch, Apparel, and Bags underperform the category carting benchmark (converting at 15.65%–19.12% vs. the 24.99% benchmark). Fixing top-of-funnel navigation to product discovery and optimizing low-converting PDP elements (imagery, sizing, pricing visibility) will unlock immediate revenue lift. 

*Note: Findings are derived from a public GA4 sample dataset (Nov 2020 – Jan 2021) and must be validated against internal telemetry before committing engineering resources.*

---

## Key Findings

1. **78.8% of site sessions drop off before viewing any product page.** Out of 354,857 total sessions, only 75,261 (21.2%) record a `view_item` event. *So what:* Users encounter severe discovery friction on entry points. In B2B SaaS, this corresponds to users logging in 

In [24]:
# ── Section 10f: Display and export product brief ─────────────────────────
from IPython.display import Markdown, display
from pathlib import Path
import pandas as pd

if brief:
    # Render Claude output as formatted markdown inside the notebook.
    display(Markdown(brief))

    # Persist to repo root for README, portfolio, or sharing outside Jupyter.
    header = (
        "# Product Brief — GA4 Funnel Analysis\n\n"
        f"*Generated {pd.Timestamp.now().strftime('%Y-%m-%d')} · Model: {BRIEF_MODEL} · "
        "Session-scoped funnel · Nov 2020 – Jan 2021 · "
        "[GA4 public e-commerce sample](https://console.cloud.google.com/marketplace/"
        "product/google-analytics-public-data/ga4_obfuscated_sample_ecommerce)*\n\n"
        "---\n\n"
    )
    out = Path("product_brief_ga4_funnel.md")
    out.write_text(header + brief, encoding="utf-8")
    print(f"Brief saved to {out}")


## Executive Summary
The primary growth bottleneck on the platform is a drop-off prior to intent: 78.8% of sessions end before a user reaches a single product detail page (PDP), and product pages in high-volume categories like YouTube merch, Apparel, and Bags underperform the category carting benchmark (converting at 15.65%–19.12% vs. the 24.99% benchmark). Fixing top-of-funnel navigation to product discovery and optimizing low-converting PDP elements (imagery, sizing, pricing visibility) will unlock immediate revenue lift. 

*Note: Findings are derived from a public GA4 sample dataset (Nov 2020 – Jan 2021) and must be validated against internal telemetry before committing engineering resources.*

---

## Key Findings

1. **78.8% of site sessions drop off before viewing any product page.** Out of 354,857 total sessions, only 75,261 (21.2%) record a `view_item` event. *So what:* Users encounter severe discovery friction on entry points. In B2B SaaS, this corresponds to users logging in (`session_start`) but failing to discover key features (`view_item`).
2. **High-volume product categories severely underperform the 24.99% view-to-cart benchmark.** PDPs for YouTube merchandise (15.65% view-to-cart), Apparel (19.12%), and Bags (17.58%) leave 1,185 cart additions on the table combined. *So what:* Product detail pages for these specific categories are failing to convert user interest into buying intent due to PDP-specific issues (e.g., pricing, missing sizing info, inadequate imagery, or out-of-stock variants).
3. **Cart-to-checkout drop-off reaches 65.0%.** Of the 14,870 sessions that reach `add_to_cart`, only 5,205 (35.0%) move to `begin_checkout`. *So what:* High-intent shoppers are abandoning at the cart stage, pointing to cart drawer friction, unexpected shipping fees, or lack of guest checkout options.
4. **Referral traffic generates the highest intent and revenue per session.** Referral sessions achieve a 30.28% product view rate, a 1.72% purchase rate, and $1.28 revenue per session—far outperforming Organic traffic (26.32% view rate, 0.90% purchase rate, $0.58 rev/session). *So what:* External link partners are sending higher-intent traffic; referral entry experiences should be protected and optimized. *(Note: CPC and `(not set)` purchase metrics were excluded from comparison due to low volume flags).*

---

## Hypotheses

1. **[HIGH] Category PDP Content & Variant Gaps:** Product detail pages for YouTube, Apparel, and Bags lack critical decision-making assets (such as clear size charts, customer reviews, detailed descriptions, or stock availability), causing product-page visitors to abandon before adding to cart.
2. **[HIGH] Homepage & Category Navigation Disconnect:** The homepage layout and global navigation fail to guide top-of-funnel traffic toward product catalog pages, driving 78.8% of sessions to exit without viewing a single product.
3. **[MEDIUM] Cart Cost Transparency Shock:** Shoppers use the cart to calculate total order cost; uncalculated shipping fees and taxes inside the cart drawer create friction, driving the 65.0% drop-off between `add_to_cart` and `begin_checkout`.

---

## Recommended Actions

1. **Optimize PDP Assets for YouTube, Apparel, and Bags Categories**
   * **What to do:** Overhaul product detail pages for these three categories by adding prominent sizing charts, expanded photo galleries, explicit stock availability, and social proof.
   * **Why:** YouTube (15.65%), Apparel (19.12%), and Bags (17.58%) represent the largest concentration of missed carts (1,185 carts at stake) against the 24.99% benchmark.
   * **Metric to watch:** Category-specific `view_to_cart_pct` (target: 25.0%+).

2. **Streamline Top-of-Nav Discovery Paths**
   * **What to do:** Redesign the homepage above-the-fold experience to feature direct category grids, personalized recommendations, and search bar prompts.
   * **Why:** 78.8% of sessions currently drop off before reaching a product page. (In SaaS terms: improving onboarding flows to move users from login directly into core feature adoption).
   * **Metric to watch:** `Session Start` -> `View Item` conversion rate (target: 30.0%+ from 21.2%).

3. **Add Upfront Cost Estimates & Quick-Checkout in the Cart**
   * **What to do:** Implement dynamic shipping calculators within the cart drawer alongside express payment options (e.g., Apple Pay, Shop Pay).
   * **Why:** 65.0% of carting sessions fail to enter checkout, indicating friction between item selection and purchase initiation.
   * **Metric to watch:** `Add to Cart` -> `Begin Checkout` conversion rate (target: 45.0%+ from 35.0%).

---

## What to Measure Next

1. **Cart Interaction & Error Logging:** Run an event-level analysis on cart drawer interactions (promo code field usage, shipping estimation triggers, line-item deletions) to pinpoint why 65.0% of cart sessions fail to begin checkout.
2. **Traffic Audit on `(not set)` Medium:** Investigate the 92,892 sessions logged as `(not set)` medium (which show a very low 6.68% `view_item` rate) to determine if this represents broken UTM tagging, dark social traffic, or web crawler bots.
3. **Cross-Sell Potential from Top-Performing Categories:** Analyze basket composition for high-converting PDP categories (e.g., Hats at 40.62% `view_to_cart` and Writing at 40.88%) to evaluate whether bundling them onto underperforming PDPs boosts total cart additions.

Brief saved to product_brief_ga4_funnel.md
